# 一般prompt
簡單針對prompt內容判斷隔日漲跌

## 從變異法(run_gemini)的輸出prompt，並比較他們的差異  
GPT-4o
Prompt1 表現比較好的組別  
Prompt2 表現比較差的組別  

語意上的差異  
措辭和語氣：  
Prompts 1 使用的措辭較為正式，重視對投資者預測和股價上漲的分析。  
Prompts 2 措辭相對更為謹慎和細緻，強調對基礎因素和假設的釐清。  

重點關注：  
Prompts 1 更加側重於投資者預期和股價增長的關係，主要關注「如何」投資者的預期能夠推動股價上漲。  
Prompts 2 更注重「為什麼」這些預期成立，強調對基礎驅動因素的深入了解和釐清。  

分析深度：  
Prompts 1 偏向於直接的影響和結果，強調對業務計劃和投資者預測的評估。  
Prompts 2 更深入地探討基礎驅動因素，強調透徹的分析和理解。  

作法上的差異  
分析方法：  
Prompts 1 使用較為直接的分析方法，側重於投資者預期和市場影響的關聯。  
Prompts 2 採取更加謹慎和深入的方法，強調對假設和基礎因素的驗證和闡明。  

重點範圍：  
Prompts 1 主要集中在投資者預期和市場反應，可能更適合於短期和直接的市場分析。  
Prompts 2 涵蓋更廣泛的範圍，包括對長期增長潛力和支持因素的詳細分析，適合於長期和深度的研究。  

語境和背景：  
Prompts 1 更加關注於投資者和市場的即時反應。  
Prompts 2 強調背景因素和深層次的驅動力，適合於構建更全面和透徹的分析框架。  

對語言模型的影響  
輸出內容的具體性：  
Prompts 1 可能會導致模型產生更為直接和結果導向的輸出。  
Prompts 2 會引導模型生成更為詳細和背景豐富的內容。  

分析的深度和廣度：  
Prompts 1 傾向於產生較為簡潔和重點明確的分析結果。  
Prompts 2 會引導模型進行更深入和廣泛的探討，涵蓋更多背景和基礎因素。  

結論的可靠性：  
Prompts 1 的結果可能較為直接，但在基礎和假設方面的細節可能不夠豐富。  
Prompts 2 的結果雖然可能更為複雜，但在細節和基礎方面會更為可靠和詳盡。  

總結來說，這兩組提示分別適合不同的分析需求和情境。Prompts 1 更適合於快速和直接的市場反應分析，而 Prompts 2 則適合於深度和長期的戰略研究。根據具體的分析目標選擇合適的提示組，可以更好地滿足不同的研究和分析需求。

## 範例
由GPT-4產生  
Prompts 1  
"全面評估一項業務創新計畫的可行性，並探討其對目標市場的影響。"  
"仔細驗證新業務計畫的各個部分，並預測其對公司股價的可能影響。"  
"深入探討業務創新計畫如何影響投資者的預期，並全面評估其長期效益。"  

Prompts 2  
"評估一項業務創新計畫如何改變目標市場的格局，並審慎評量其實施的風險。"   
"分析新業務計畫對公司股價的可能影響，並審慎評量其對投資者的吸引力。"  
"探討業務創新計畫如何影響投資者的預期回報，並審慎評量其對公司財務狀況的影響。"  

In [86]:
# %pip install langchain_google_genai
# %pip install mplfinance

In [87]:
import os
import textwrap
import math
import pandas as pd
import numpy as np

from datetime import datetime
from dateutil.relativedelta import relativedelta

import mplfinance as mpf
import google.generativeai as genai
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory

from IPython.display import display
from IPython.display import Markdown

model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

In [88]:
import pandas as pd
import json
import os

def eval(out_file):
    # 計算準確率和平均報酬率
    gain = []
    loss = []

    tp = 0
    fp = 0
    tn = 0
    fn = 0

    token_input = 0
    token_output = 0

    path_price_his = os.path.dirname(os.path.abspath(os.getcwd())) + "/history_data/tw/twii_history.csv"
    df_price_his = pd.read_csv(path_price_his, encoding='utf-8')
    df_price_his['日期'] = pd.to_datetime(df_price_his['日期'], format='%Y%m%d')

    with open(out_file, "r", encoding='utf-8') as f:
        data = json.load(f)
        output_data = data["output_data"]

        for date_str, result in output_data.items():
            token_input += result["token"]["input_tokens"]
            token_output += result["token"]["output_tokens"]
            sig_str = result["raw_data"]
            sig = 0

            if "unknown" in sig_str:
                sig = 0
            elif "yes" in sig_str:
                sig = 1
            elif "no" in sig_str:
                sig = -1
            else:
                sig = 0

            df_price = df_price_his[df_price_his['日期'] == pd.to_datetime(date_str, format='%Y%m%d')]

            if df_price.empty:
                continue

            rtn = (df_price.iloc[0]['收盤指數'] - df_price.iloc[0]['開盤指數']) / df_price.iloc[0]['開盤指數']

            if pd.isna(rtn):
                rtn = 0

            if sig > 0:
                if rtn > 0:
                    tp += 1
                    gain.append(rtn)
                else:
                    fp += 1
                    loss.append(rtn)
            elif sig == -1:
                if rtn < 0:
                    tn += 1
                    gain.append(-rtn)
                else:
                    fn += 1
                    loss.append(-rtn)

    print(f'tp {tp}, fp {fp}, tn {tn}, fn {fn}')
    if tp + fp + tn + fn == 0:
        accuracy = 0
        print(f'Accuracy {accuracy}, Expected value:{0}')
    else:
        accuracy = (tp + tn) / (tp + fp + tn + fn)
        print(f'Accuracy {accuracy}, Expected value:{(sum(gain) / len(gain))*accuracy + (sum(loss) / len(loss))*(1-accuracy)}')
    print("--------------------")
    print(f'token: input {token_input}, output {token_output}')
    print(f'預估費用 {(token_input/1000000)*0.35 + (token_output/1000000)*0.7} USD, {((token_input/1000000)*0.35 + (token_output/1000000)*0.7)*32.6} TWD')
    print(f'詢問次數 {len(output_data)}')

    return accuracy

In [89]:
import time
import os
import json
import pandas as pd

def step1(stock_ids, st, et, init_prompt):
    out_path = "out_prompt_tx/"
    count = 1
    while True:
        if os.path.exists(out_path + str(count) + ".json"):
            count += 1
            continue
        else:
            out_file = out_path + str(count) + ".json"
            break

    path_price_his = os.path.dirname(os.path.abspath(os.getcwd())) + "/history_data/tw/twii_history.csv"

    # 讀取CSV檔案
    df_price_his = pd.read_csv(path_price_his, encoding='utf-8')
    df_price_his['日期'] = pd.to_datetime(df_price_his['日期'], format='%Y%m%d')
    df_price_his = df_price_his[(st <= df_price_his['日期']) & (df_price_his['日期'] <= et)]

    idx = 1
    batch_size = 15
    output_data = {}  # 使用字典來存儲每一天的輸出數據

    while idx < len(df_price_his):
        batch = []
        for i in range(batch_size):
            if idx + i >= len(df_price_his):
                break

            news_date = df_price_his.iloc[idx - i]['日期'] # 取得前一日的日期

            # 存儲所有股票的新聞資料
            text_news = ""
            for j, stock_id in enumerate(stock_ids):
                path_news_file = os.path.dirname(os.path.abspath(os.getcwd())) + "/history_data/tw/news_title/" + stock_id + "news_title.json"
                with open(path_news_file, 'r', encoding='utf-8') as f:
                    news_data = json.load(f)
                    text_news += f" 股票{j+1}的熱門新聞標題: {news_data[news_date.strftime('%Y%m%d')]} \n"

            # 設置初始消息列表，包括 SystemMessage 和第一個 HumanMessage
            messages = [
                ("system", "你是一個厲害的股市操盤手，專長是當日沖消"),
                ("human",
                 f"""以下是台灣股市權值股的新聞標題，{text_news}
                 {init_prompt}
                 請根據上述步驟判斷台指期隔日開盤時買進收盤時賣出的獲利可能性，如果可以獲利請只回答#yes，否則只回答#no，如果與股市漲跌無關則回答#unknown，請盡量不要回答 #unknown
                 """)
            ]
            batch.append(messages)

        res = llm.batch(batch)

        for i in range(len(res)):
            date_str = df_price_his.iloc[idx]['日期'].strftime('%Y%m%d')
            output_data[date_str] = {
                "token": res[i].usage_metadata,
                "raw_data": res[i].content,
            }

        time.sleep(60)
        idx += batch_size

    # 將所有輸出數據寫入到同一個json檔案中
    data = {
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "init_prompt": init_prompt,
        "output_data": output_data,
    }
    json_content = json.dumps(data, ensure_ascii=False, indent=4)
    with open(out_file, "w", encoding="UTF-8") as f:
        f.write(json_content)

    return out_file

多層次的綜合評估

步驟一：宏觀環境與行業動態分析

首先，對公司所在行業的宏觀經濟環境進行全面研究，考察全球及區域性的經濟趨勢、政策變動、技術進步和社會變遷等因素。深入了解行業內外的競爭格局、主要趨勢及變化，包括技術創新、資本流動、政策支持等，從而確定行業的整體發展潛力。

步驟二：公司策略與資源評估

接著，深入評估公司的策略和資源配置，包括其創新戰略、資本投入、人力資源配置及研發能力。分析公司在新產品開發、市場拓展和數字化轉型等方面的具體措施，並評估這些措施的可行性及潛在影響。考察公司的領導力、管理層的決策能力及其執行效率，從而了解公司是否具備持續創新和實現長期增長的潛力。

步驟三：市場需求與競爭環境調查

深入調查目標市場的需求變化和競爭環境，包括特定年齡層、地理區域或行業的消費者行為及偏好。通過市場調研、數據分析和競品對比，了解市場對新產品和服務的需求，並評估競爭對手的反應及市場份額的變化。結合消費者行為的趨勢，分析公司的創新舉措是否能夠滿足市場需求並取得競爭優勢。

步驟四：創新舉措的財務影響評估

評估創新舉措的財務回報，包括對公司收入增長、成本控制、利潤率等方面的影響。建立財務模型，模擬不同創新方案在各種市場條件下的預期回報，分析其對公司整體財務健康狀況的影響。考察公司的資本結構、資金運營效率及風險管理能力，以確保創新舉措的穩定實施和長期收益。

步驟五：投資者預期與股價影響分析

最後，研究投資者對公司創新舉措的預期回報，並分析這些預期如何轉化為股價的變動。通過分析市場反應、投資者情緒及交易行為，找出影響股價的主要驅動因素。結合宏觀經濟趨勢、行業前景、公司內部創新能力和執行力等因素，綜合評估創新舉措對公司長期股價表現的潛在影響


In [91]:
# 權值股股票代號
# 代號 ['2308', '2317', '2330', '2382', '2412', '2454', '2881', '2882', '2891', '6505']

from datetime import datetime

start_day = datetime(2023, 6, 1)
end_day = datetime(2024, 5, 30)

stock_ids = ['2308', '2317', '2330', '2382', '2412', '2454', '2881', '2882', '2891', '6505']
init_acc = 0

init_prompt = """
評估創新舉措的財務回報，包括對公司收入增長、成本控制、利潤率等方面的影響。建立財務模型，模擬不同創新方案在各種市場條件下的預期回報，分析其對公司整體財務健康狀況的影響。考察公司的資本結構、資金運營效率及風險管理能力，以確保創新舉措的穩定實施和長期收益。
"""
print(f"start")
outpath = step1(stock_ids, start_day, end_day , init_prompt)
acc = eval(outpath)
print(f"step1 done")
print()

start


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


tp 0, fp 0, tn 0, fn 0
Accuracy 0, Expected value:0
--------------------
token: input 624301, output 476
預估費用 0.21883855 USD, 7.13413673 TWD
詢問次數 238
step1 done



In [ ]:
#計算準確率
import pandas as pd
import json
import os

out_path = "out_prompt_tx/11/"
files = os.listdir(out_path)
files = sorted(files)

for idx, file in enumerate(files):
    # print(file)
    sig = 0
    with open(out_path + file, "r", encoding='utf-8') as f:
        data = json.load(f)

        for i in range(len(res)):
            date_str = df_price_his.iloc[idx]['日期'].strftime('%Y%m%d')
            output_data[date_str] = {
                "token": res[i].usage_metadata,
                "raw_data": res[i].content,
            }

        time.sleep(60)
        idx += batch_size

    # 將所有輸出數據寫入到同一個json檔案中
    data = {
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "init_prompt": init_prompt,
        "output_data": output_data,
    }
    json_content = json.dumps(data, ensure_ascii=False, indent=4)
    with open(out_file, "w", encoding="UTF-8") as f:
        f.write(json_content)

    return out_file

In [ ]:
#計算準確率
import pandas as pd
import json
import os

path_price_his = os.path.dirname(os.path.abspath(os.getcwd())) + "/history_data/tw/twii_history.csv"
df_price_his = pd.read_csv(path_price_his)
df_price_his['日期'] = pd.to_datetime(df_price_his['日期'], format='%Y%m%d')
token_input = 0
token_output = 0


# 計算準確率和平均報酬率
gain = []
loss = []

tp = 0
fp = 0
tn = 0
fn = 0

out_path = "out_prompt_tx/11/"
files = os.listdir(out_path)
files = sorted(files)

for idx, file in enumerate(files):
    # print(file)
    sig = 0
    with open(out_path + file, "r", encoding='utf-8') as f:
        data = json.load(f)
        token_input += data["token"]["input_tokens"]
        token_output += data["token"]["output_tokens"]
        sig_str = data["raw_data"]

        if sig_str.find("unknown") > 0:
            sig = 0
        elif sig_str.find("yes") > 0:
            sig = 1
        elif sig_str.find("no")>0:
            sig = -1
        else:
            sig = 0
        print(sig, sig_str)


    if idx+1 == len(files):
        break
    
    # print(files[idx+1])

    date_str = (files[idx+1].split(".")[0]).split("out")[1]
    df_price = df_price_his[df_price_his['日期'] == date_str]

    rtn = (df_price.iloc[0]['收盤指數'] - df_price.iloc[0]['開盤指數']) / df_price.iloc[0]['開盤指數']

    if pd.isna(rtn):
        rtn = 0

    if sig > 0:
        if rtn > 0:
            tp += 1
            gain.append(rtn)
        else:
            fp += 1
            loss.append(rtn)
    elif sig == -1: 
        if rtn < 0:
            tn += 1
            gain.append(-rtn)
        else:
            fn += 1
            loss.append(-rtn)

# print(max_gain, max_loss, tp+fp+tn+fn)
print(f'tp {tp}, fp {fp}, tn {tn}, fn {fn}')
accuracy = (tp + tn) / (tp + fp + tn + fn)
print(f'Accuracy:{accuracy}, Precision:{tp / (tp + fp)}, Expected value:{(sum(gain) / len(gain))*accuracy + (sum(loss) / len(loss))*(1-accuracy)}')
print(f'Average gain: {sum(gain) / len(gain)}, Average loss: {sum(loss) / len(loss)}')
print(f'詢問次數 {len(files)}')
print()
    # print(tn+tp+fn+fp)
print("--------------------")
print(f'token: input {token_input}, output {token_output}')
print(f'累計預估費用 {(token_input/1000000)*0.35 + (token_output/1000000)*1.05} USD, {((token_input/1000000)*0.35 + (token_output/1000000)*1.05)*32.6} TWD')
